# TCRW Fig. 3 Artifact Audit

This notebook documents the completed Fig. 3 reproduction using the existing plots, summary files, code, and overlay diagnostics. It does not rerun the expensive Fortran drivers or remake the panel plots.



## Reader Guide: What Fig. 3 Is Teaching

Fig. 3 is the **quantitative steady-state figure**. It turns the visual ideas from Fig. 2 into parameter scans.

How to read it:

- **Panels (a,f):** edge localization strength. These ask how $P_{edge}/P_{bulk}$ changes with $D_r$ or $\omega$.
- **Panels (b,g):** relative current components. These ask whether noise-attributed current or chirality-attributed current dominates.
- **Panels (c,d,h,i):** left-wall vector fields. Arrow direction is current direction; color is current magnitude.
- **Panels (e,j):** angle summaries. These compress many wall vectors into a single orientation observable.

The most important audit lesson is panel **(d)**. The exact steady-state $J_\omega$ field is smooth near small $D_r$, but finite Fortran MC current counts are rough there. The no-shift overlay proves this is sampling roughness, not a wrong transition matrix or plotting convention.

Two angle conventions matter:

- **Full-wall totals** include corner sites and are useful for exact-vs-Fortran cross-checks.
- **Interior-only totals** drop corner contamination and match the paper-style sharp angle transitions in panels (e) and (j).


In [1]:
from __future__ import annotations
from pathlib import Path
import os
import textwrap
import numpy as np

ROOT = Path.cwd()

def data_rows(path):
    with open(path) as f:
        return sum(1 for line in f if line.strip() and not line.startswith("#"))

def artifact_report(paths):
    print("artifact                                  exists  size_MB  modified")
    for raw in paths:
        p = ROOT / raw
        exists = p.exists()
        size = p.stat().st_size / 1024 / 1024 if exists else float("nan")
        mtime = __import__("datetime").datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M") if exists else "-"
        print(f"{raw:42s} {str(exists):>6s} {size:8.3f}  {mtime}")

def source_head(path, n=70):
    p = ROOT / path
    print(f"--- {path} ---")
    with open(p, "r", errors="replace") as f:
        for i, line in zip(range(n), f):
            print(line.rstrip())


## Existing Original Plots And Diagnostics

### Author-code / Python Exact Plot

![Fig3 authors exact](tcrw_fig3_authors.png)

### Python MC/Reference Plot

![Fig3 Python MC](tcrw_fig3_pymc.png)

### Paper Comparison

![Fig3 comparison](tcrw_fig3_compare_with_paper.png)

### Final No-Shift cdhi Overlay

![Fig3 cdhi no-shift overlay](tcrw_fig3_cdhi_overlay_noshift.png)


## What You Did For Fig. 3

1. Read the paper and authors' original code to pin down the transition matrix and current decomposition.
2. Built exact Python reference code in `tcrw_fig3_exact.py` and plotting helper `_fig3_plot.py`.
3. Built Fortran drivers for panels (a), (b), (c,d,e), (f), (g), and (h,i,j).
4. Dropped the singular $D_r=1$ endpoint in the $D_r$ scans to avoid the degenerate no-translation limit.
5. Added normalized current columns to cde/hij Fortran summaries so arrow colors/lengths can be audited consistently.
6. Preserved full-wall total angles for Python-vs-Fortran cross-checks, but added interior-only angle columns for paper-faithful panels (e) and (j).
7. Fixed gnuplot c/d/h/i arrow conventions to align with Python exact plotting: fixed arrow length for c/d/i, magnitude-scaled length for h.
8. Created `tcrw_fig3_cdhi_overlay.py` and the no-shift overlay to compare exact steady-state arrows against Fortran MC arrows on the same grid.
9. Resolved the apparent panel (d) mismatch: exact $J_\omega$ is smooth near $D_r=10^{-4}$, while Fortran MC arrows are rough because current counts are noisy at tiny $D_r$.


## Code Map

| Panel | Fortran/Python source | Data/plot artifacts | Audit point |
|---|---|---|---|
| 3(a) | `tcrw_fig3a.f90`, `tcrw_fig3a_crosscheck.py` | `tcrw_fig3a_summary.txt`, `tcrw_fig3a.pdf` | Edge/bulk density ratio vs $D_r$ |
| 3(b) | `tcrw_fig3b.f90`, `tcrw_fig3b_crosscheck.py` | `tcrw_fig3b_summary.txt`, `tcrw_fig3b.pdf` | Wall-current ratio vs $D_r$ |
| 3(c,d,e) | `tcrw_fig3cde.f90`, `tcrw_fig3cde_crosscheck.py` | `tcrw_fig3cde_summary.txt`, `tcrw_fig3e_summary.txt` | Left-edge currents and angles vs $D_r$ |
| 3(f) | `tcrw_fig3f.f90`, `tcrw_fig3f_crosscheck.py` | `tcrw_fig3f_summary.txt`, `tcrw_fig3f.pdf` | Edge/bulk density ratio vs $\omega$ |
| 3(g) | `tcrw_fig3g.f90`, `tcrw_fig3g_crosscheck.py` | `tcrw_fig3g_summary.txt`, `tcrw_fig3g.pdf` | Wall-current ratio vs $\omega$ |
| 3(h,i,j) | `tcrw_fig3hij.f90`, `tcrw_fig3hij_crosscheck.py` | `tcrw_fig3hij_summary.txt`, `tcrw_fig3j_summary.txt` | Left-edge currents and angles vs $\omega$ |
| Exact reference | `TRW._original_code_by_paperauthors.py`, `tcrw_fig3_exact.py`, `tcrw_fig3_authors.py` | `tcrw_fig3_authors.png` | Scientific reference |
| MC style | `tcrw_fig3_pymc.py` and Fortran summaries | `tcrw_fig3_pymc.png`, gnuplot PDFs | Paper-style stochastic visual match |


In [2]:
artifact_report([
    "tcrw_fig3a.pdf", "tcrw_fig3b.pdf", "tcrw_fig3c.pdf", "tcrw_fig3d.pdf", "tcrw_fig3e.pdf",
    "tcrw_fig3f.pdf", "tcrw_fig3g.pdf", "tcrw_fig3h.pdf", "tcrw_fig3i.pdf", "tcrw_fig3j.pdf",
    "tcrw_fig3_authors.png", "tcrw_fig3_pymc.png", "tcrw_fig3_compare_with_paper.png",
    "tcrw_fig3_cdhi_overlay.png", "tcrw_fig3_cdhi_overlay_noshift.png",
    "tcrw_fig3cde_crosscheck.png", "tcrw_fig3hij_crosscheck.png",
])


artifact                                  exists  size_MB  modified
tcrw_fig3a.pdf                               True    0.051  2026-04-30 23:05
tcrw_fig3b.pdf                               True    0.068  2026-04-30 23:05
tcrw_fig3c.pdf                               True    0.073  2026-04-30 23:47
tcrw_fig3d.pdf                               True    0.070  2026-04-30 23:45
tcrw_fig3e.pdf                               True    0.066  2026-04-30 23:05
tcrw_fig3f.pdf                               True    0.058  2026-04-30 23:05
tcrw_fig3g.pdf                               True    0.069  2026-04-30 23:05
tcrw_fig3h.pdf                               True    0.059  2026-04-30 23:45
tcrw_fig3i.pdf                               True    0.061  2026-04-30 23:45
tcrw_fig3j.pdf                               True    0.059  2026-04-30 23:05
tcrw_fig3_authors.png                        True    0.586  2026-04-30 23:46
tcrw_fig3_pymc.png                           True    0.585  2026-04-30 22:12
tcrw_fig

In [3]:
print("Summary shapes")
for name in [
    "tcrw_fig3a_summary.txt", "tcrw_fig3b_summary.txt", "tcrw_fig3cde_summary.txt", "tcrw_fig3e_summary.txt",
    "tcrw_fig3f_summary.txt", "tcrw_fig3g_summary.txt", "tcrw_fig3hij_summary.txt", "tcrw_fig3j_summary.txt",
]:
    arr = np.loadtxt(name, comments="#")
    print(f"{name:28s} shape={arr.shape} x-range={arr[:,1].min():.5g}..{arr[:,1].max():.5g}")


Summary shapes
tcrw_fig3a_summary.txt       shape=(100, 7) x-range=0.0001..0.97724
tcrw_fig3b_summary.txt       shape=(100, 9) x-range=0.0001..0.97724
tcrw_fig3cde_summary.txt     shape=(275, 14) x-range=0.0001..0.97724
tcrw_fig3e_summary.txt       shape=(25, 14) x-range=0.0001..0.97724
tcrw_fig3f_summary.txt       shape=(63, 7) x-range=0..1
tcrw_fig3g_summary.txt       shape=(63, 9) x-range=0..1
tcrw_fig3hij_summary.txt     shape=(231, 14) x-range=0..1
tcrw_fig3j_summary.txt       shape=(21, 14) x-range=0..1


## Final cdhi Overlay Result

The decisive diagnostic was the no-shift overlay:

- Gray arrows: exact steady-state field from the authors' transition matrix.
- Colored arrows: Fortran MC current field from the fresh summary files.
- Same grid points; gray arrows drawn slightly longer and colored arrows slightly shorter.

Key numerical result recorded from the overlay run:

| Panel | Median $|\Delta\theta|$ | 90th percentile | Max | Interpretation |
|---|---:|---:|---:|---|
| c | $5.25\times10^{-4}$ rad | $2.85\times10^{-3}$ rad | $2.55\times10^{-2}$ rad | Good overlap |
| d | $2.11\times10^{-2}$ rad | $2.63\times10^{-1}$ rad | $5.88\times10^{-1}$ rad | Rough at tiny $D_r$ from MC noise |
| h | $4.59\times10^{-4}$ rad | $2.23\times10^{-3}$ rad | $8.57\times10^{-3}$ rad | Good away from $\omega=0.5$ zero-current point |
| i | $1.08\times10^{-2}$ rad | $4.57\times10^{-2}$ rad | $2.30\times10^{-1}$ rad | Moderate MC jitter |

At the smallest $D_r=10^{-4}$ in panel (d): exact $\theta_{J_\omega}=\pi/4$ across the wall, while the MC angle range was roughly $0.263$ to $1.320$ rad. That explains why paper-style arrows look rough while the exact solution is smooth.


## Source Headers


In [4]:
for fname in ["tcrw_fig3a.f90", "tcrw_fig3b.f90", "tcrw_fig3cde.f90", "tcrw_fig3hij.f90", "tcrw_fig3_cdhi_overlay.py"]:
    source_head(fname, n=45)


--- tcrw_fig3a.f90 ---
!=====================================================================
! tcrw_fig3a.f90 — Fig 3(a): P_edge/P_bulk vs D_r for ω = 1, multiple L
!
! Paper: Osat, Meyberg, Metson, Speck,
!        "Topological chiral random walker"
!        arXiv:2602.12020v1 [cond-mat.stat-mech], 12 Feb 2026.
! Panel: Fig 3(a) — log-log plot of the per-site edge/bulk ratio of the
!        steady-state occupation probability vs. the rotational noise
!        rate D_r, at maximal chirality ω = 1, for system sizes
!        L ∈ {4, 9, 19, 49}.  The paper goes up to L = 99 (and higher)
!        using exact diagonalization; MC at L = 99 with the small-D_r
!        end of the range would need T ≳ 10^9 for a clean ratio, so we
!        stop at L = 49 here.
!        Caption: ratio collapses onto a master curve, independent of L
!                 once normalized per site → "edge localization is a
!                 bulk-band-topology effect, not a finite-size artefact".
!
! Convention (matches

## Fig. 3 Verdict

- The main Fig. 3 reproduction is complete.
- Python exact is the clean scientific reference.
- Fortran MC explains the paper-style rough arrows, especially panel (d) at tiny $D_r$.
- The overlay proves the mismatch is finite-sampling roughness, not a wrong transition matrix or current convention.
